In [29]:
import os
from openai import OpenAI
import pandas as pd
from enum import Enum
from pydantic import BaseModel
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import json

from dotenv import load_dotenv
load_dotenv()


True

In [30]:
"""from nltk.tokenize import PunktTokenizer

sent_detector = PunktTokenizer("portuguese")

def sentencize_df(df, text_col, id_col=None):
    rows = []
    for i, row in df.iterrows():
        text = str(row[text_col])
        text = "\n".join([line.strip() for line in text.split("\n") if line.strip()])
        sentences = sent_detector.tokenize(text)
        for j, sent in enumerate(sentences):
            entry = {
                "article_idx": i,
                "sentence_idx": j,
                "sentence": sent,
            }
            if id_col:
                entry[id_col] = row[id_col]
            rows.append(entry)
    return pd.DataFrame(rows)
"""

'from nltk.tokenize import PunktTokenizer\n\nsent_detector = PunktTokenizer("portuguese")\n\ndef sentencize_df(df, text_col, id_col=None):\n    rows = []\n    for i, row in df.iterrows():\n        text = str(row[text_col])\n        text = "\n".join([line.strip() for line in text.split("\n") if line.strip()])\n        sentences = sent_detector.tokenize(text)\n        for j, sent in enumerate(sentences):\n            entry = {\n                "article_idx": i,\n                "sentence_idx": j,\n                "sentence": sent,\n            }\n            if id_col:\n                entry[id_col] = row[id_col]\n            rows.append(entry)\n    return pd.DataFrame(rows)\n'

In [31]:
portuguese_df = pd.read_csv("FakeTrueBr_corpus.csv")
english_df = pd.read_csv("task3-df.csv")

In [32]:
portuguese_df["fake"].str.split().str.len().describe()

count    1791.000000
mean      143.862647
std       116.274896
min        17.000000
25%        69.000000
50%       109.000000
75%       179.000000
max      1241.000000
Name: fake, dtype: float64

In [33]:
# artigo com mais caracteres
idx = portuguese_df["fake"].str.len().idxmax()
print(portuguese_df.loc[idx, "title_fake"])

renan calheiros grava áudio em que diz que mourão e stf armam golpe contra bolsonaro  


In [34]:
portuguese_df.columns

Index(['title_fake', 'fake', 'link_f', 'true', 'link_t'], dtype='str')

In [35]:
english_df.columns

Index(['article_id', 'text_id', 'label', 'text'], dtype='str')

In [36]:
portuguese_df["fake"].iloc[0]

'carnaval em olinda. arrastão monstro. fazuele isso foi só um aperitivo do que tá por vir no carnaval país comandado por bandidos, dá nisso. , 172, sexta feira de carnaval, em olinda os pobres meninos do luladrão promovendo um arrastão, nas ruas de olinda, atacando turistas para roubar um ou outro celular e ter um dinheirinho prá tomar uma cervejinha parecem mais uma praga de gafanhotos atacando uma plantação! o brasil está entregue à criminalidade com a proteção do presidente!  o amor venceu '

In [37]:
len(portuguese_df)

1791

In [38]:
english_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12291 entries, 0 to 12290
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   article_id  12291 non-null  int64
 1   text_id     12291 non-null  int64
 2   label       4713 non-null   str  
 3   text        12291 non-null  str  
dtypes: int64(2), str(2)
memory usage: 384.2 KB


In [39]:
english_df_cleaned = english_df[english_df['label'].notna()]


In [40]:
(
    english_df_cleaned['label']
    .dropna()
    .str.split(',')      # "A,B" → ["A", "B"]
    .explode()           # uma linha por label
    .str.strip()         # remove espaços acidentais
    .value_counts()
)

label
Loaded_Language                    2210
Name_Calling-Labeling              1184
Doubt                               679
Repetition                          657
Exaggeration-Minimisation           554
Appeal_to_Fear-Prejudice            433
Flag_Waving                         376
Causal_Oversimplification           227
False_Dilemma-No_Choice             182
Appeal_to_Authority                 174
Slogans                             173
Conversation_Killer                 113
Red_Herring                          61
Guilt_by_Association                 59
Appeal_to_Popularity                 47
Appeal_to_Hypocrisy                  46
Obfuscation-Vagueness-Confusion      31
Straw_Man                            22
Whataboutism                         17
Name: count, dtype: int64

In [41]:
PROPAGANDA_LABELS = [
    "Linguagem_Carregada",
    "Xingamento-Rotulação",
    "Dúvida",
    "Repetição",
    "Exagero-Minimização",
    "Apelo_ao_Medo-Preconceito",
    "Apelo_à_Bandeira",
    "Simplificação_Causal",
    "Falso_Dilema-Sem_Escolha",
    "Apelo_à_Autoridade",
    "Slogans",
    "Encerrador_de_Debate",
    "Arenque_Vermelho",
    "Culpa_por_Associação",
    "Apelo_à_Popularidade",
    "Apelo_à_Hipocrisia",
    "Ofuscação-Vagueza-Confusão",
    "Espantalho",
    "Whataboutismo",
    "Nenhuma",
]

In [42]:
LABEL_MAP_PT_EN = {
    "Linguagem_Carregada": "Loaded_Language",
    "Xingamento-Rotulação": "Name_Calling-Labeling",
    "Dúvida": "Doubt",
    "Repetição": "Repetition",
    "Exagero-Minimização": "Exaggeration-Minimisation",
    "Apelo_ao_Medo-Preconceito": "Appeal_to_Fear-Prejudice",
    "Apelo_à_Bandeira": "Flag_Waving",
    "Simplificação_Causal": "Causal_Oversimplification",
    "Falso_Dilema-Sem_Escolha": "False_Dilemma-No_Choice",
    "Apelo_à_Autoridade": "Appeal_to_Authority",
    "Slogans": "Slogans",
    "Encerrador_de_Debate": "Conversation_Killer",
    "Arenque_Vermelho": "Red_Herring",
    "Culpa_por_Associação": "Guilt_by_Association",
    "Apelo_à_Popularidade": "Appeal_to_Popularity",
    "Apelo_à_Hipocrisia": "Appeal_to_Hypocrisy",
    "Ofuscação-Vagueza-Confusão": "Obfuscation-Vagueness-Confusion",
    "Espantalho": "Straw_Man",
    "Whataboutismo": "Whataboutism",
    "Nenhuma": "None",
}

In [43]:
SYSTEM_PROMPT = """Você é um especialista em propaganda política e técnicas de persuasão.

Sua tarefa: dado um texto em português, identifique quais técnicas de propaganda estão sendo usadas.
Você deve retornar um JSON com o campo "labels" contendo uma lista de técnicas dentre:

{labels}

Definições:
- Apelo_à_Autoridade: afirmar que algo é verdade simplesmente porque uma autoridade ou especialista disse, sem outras evidências
- Apelo_à_Popularidade: afirmar que algo é verdadeiro ou bom porque muitas pessoas acreditam nisso (falácia do senso comum)
- Apelo_ao_Medo-Preconceito: construir apoio instilando ansiedade ou pânico em relação a uma alternativa, frequentemente com avisos exagerados
- Apelo_à_Bandeira: explorar sentimento nacional ou de grupo para justificar ou promover uma ação ou ideia
- Simplificação_Causal: assumir uma única causa quando há múltiplas causas para um problema
- Falso_Dilema-Sem_Escolha: apresentar apenas duas opções ignorando outras alternativas
- Espantalho: deturpar o argumento de alguém para facilitar o ataque
- Arenque_Vermelho: introduzir material irrelevante para desviar a atenção do assunto principal
- Whataboutismo: desviar críticas apontando faltas semelhantes do oponente
- Slogans: frases breves e marcantes que funcionam como atalhos emocionais, frequentemente com estereótipos
- Encerrador_de_Debate: frases que encerram o debate afirmando que o assunto é óbvio demais para discutir
- Linguagem_Carregada: palavras ou frases com forte conotação emocional usadas para influenciar o público
- Repetição: repetir a mesma mensagem para que o público gradualmente a aceite
- Exagero-Minimização: representar algo de forma excessiva ou tornar algo menos importante do que realmente é
- Ofuscação-Vagueza-Confusão: usar linguagem confusa ou vaga para obscurecer o ponto
- Xingamento-Rotulação: atribuir um rótulo a uma pessoa ou entidade para intimidar ou associá-la a algo positivo ou negativo
- Dúvida: questionar a credibilidade de alguém ou algo sem evidências
- Culpa_por_Associação: desacreditar uma pessoa associando-a a um grupo ou ideia com conotação negativa
- Apelo_à_Hipocrisia: desacreditar um argumento alegando que quem o faz é culpado da mesma coisa (tu quoque)
- Nenhuma: nenhuma técnica de propaganda detectada

Retorne apenas JSON: {{"labels": [...], "justification": "..."}}
""".format(labels=", ".join(PROPAGANDA_LABELS))

In [51]:
class LLMLabeler:
    def __init__(self, backend: str = "ollama", model: str = "qwen2.5:7b", 
                 few_shot_df: Optional[pd.DataFrame] = None,
                 n_shots: int = 3):
        self.model = model
        self.backend = backend
        self.system_prompt = SYSTEM_PROMPT
        self.few_shot_examples = self._build_few_shot(few_shot_df, n_shots)
        self._client = self._init_client()

    def _init_client(self):
        if self.backend == "ollama":
            import ollama
            return ollama
        elif self.backend == "openai":
            return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    def _build_few_shot(self, df, n_shots):
        """Sample n_shots labeled examples from English dataset per label."""
        if df is None:
            return []
        examples = []
        labeled = df[df['label'].notna()].copy()
        for label in PROPAGANDA_LABELS[:5]:  # top labels only
            subset = labeled[labeled['label'].str.contains(label, na=False)]
            for _, row in subset.sample(min(n_shots, len(subset))).iterrows():
                examples.append({
                    "text": row['text'],
                    "labels": row['label'].split(','),
                })
        return examples

    def _build_messages(self, text: str) -> list[dict]:
        messages = [{"role": "system", "content": self.system_prompt}]
        # Add few-shot examples
        for ex in self.few_shot_examples[:5]:
            messages.append({"role": "user", "content": ex["text"]})
            messages.append({"role": "assistant", "content":
                f'{{"labels": {json.dumps(ex["labels"])}, "justification": "example"}}'})
        messages.append({"role": "user", "content": text})
        return messages

    def label(self, text: str) -> dict:
        messages = self._build_messages(text)
        try:
            if self.backend == "ollama":
                response = self._client.chat(
                    model=self.model,
                    messages=messages,
                    format="json"
                )
                return json.loads(response['message']['content'])
            elif self.backend == "openai":
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    response_format={"type": "json_object"},
                )
                return json.loads(response.choices[0].message.content)
        except Exception as e:
            return {"labels": [], "justification": f"ERROR: {e}"}

    def label_dataframe(self, df: pd.DataFrame, text_col: str,
                        id_col: Optional[str] = None,
                        max_workers: int = 1) -> pd.DataFrame:
        def _label_row(args):
            idx, row = args
            out = self.label(row[text_col])
            result = {
                "labels": ",".join(out.get("labels", [])),
                "justification": out.get("justification", ""),
                "text": row[text_col],
            }
            if id_col:
                result[id_col] = row[id_col]
            return idx, result

        rows = list(df.iterrows())
        results = [None] * len(rows)

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(_label_row, item): i for i, item in enumerate(rows)}
            for future in tqdm(as_completed(futures), total=len(futures)):
                i = futures[future]
                _, result = future.result()
                results[i] = result

        return pd.DataFrame(results)


Translate labels from english_df_cleaned

In [45]:
EN_PT = {v: k for k, v in LABEL_MAP_PT_EN.items()}  # inverte o mapa EN→PT

def translate_label(label_str):
    if pd.isna(label_str):
        return label_str
    labels = [l.strip() for l in label_str.split(",")]
    translated = [EN_PT.get(l, l) for l in labels]  # mantém original se não achar
    return ",".join(translated)

english_df_cleaned["label_pt"] = english_df_cleaned["label"].apply(translate_label)

In [46]:
sample_df = portuguese_df.head(3)

Using it

In [52]:
labeler = LLMLabeler(backend="openai", model="gpt-4o-mini", few_shot_df=english_df_cleaned)

In [53]:
labeled_df = labeler.label_dataframe(df=portuguese_df, text_col="fake", max_workers=10)
labeled_df.head()
labeled_df.to_csv("labeled_df.csv", index=False)

100%|██████████| 1791/1791 [14:32<00:00,  2.05it/s]


In [54]:
labeled_df

,labels,justification,text
0,"Xingamento-Rotulação,Apelo_ao_Medo-Preconceito...",O texto utiliza a rotulação ao chamar Lula de ...,carnaval em olinda. arrastão monstro. fazuele ...
1,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza o medo ao mencionar que há 'ca...,carro alegórico da escola de samba grande rio...
2,"Xingamento-Rotulação,Dúvida,Apelo_ao_Medo-Prec...",O texto menciona 'cantor léo apoiando atos ant...,cantor léo apoiando atos antidemocráticos. alg...
3,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza apelo ao medo ao descrever a g...,versão 1 ore muito pela cantora anitta! ela es...
4,,O texto apresenta informações factuais sobre a...,morre erasmo carlos aos 81 anos cantor conside...
...,...,...,...
1786,"Apelo_ao_Medo-Preconceito,Repetição,Xingamento...",O texto contém várias técnicas de propaganda. ...,"- bom dia, eu to aqui no paraguai. estamos aq..."
1787,"Apelo_ao_Medo-Preconceito,Xingamento-Rotulação...",O texto utiliza o Apelo ao Medo ao sugerir que...,pesquisem quem e michael yeadon. e pesquisam o...
1788,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza o Apelo ao Medo ao instigar an...,na vacinacao do covid e super importante olhar...
1789,"Apelo_ao_Medo-Preconceito,Culpa_por_Associação...",O texto utiliza apelo ao medo ao sugerir que a...,lider indigina morre apos tomar vachina! a mor...
